# Notebook 02: Classification Heads on Frozen Encoder

Goal of this notebook: Evaluate different classification head architectures on top of the Moirai foundation model. 

We evaluate four distinct architectures:
1. Mean Pooling
2. Single-Scale Attention
3. Single-Scale Multi-Head Attention (MHA)
4. Hierarchical Multi-Head Attention

In [2]:
import torch
import pandas as pd
from IPython.display import display

from uni2ts.model.moirai import MoiraiModule
from moirai_classification.encoder import MoiraiEncoder
from moirai_classification.heads import (
    MeanPoolingClassifier, 
    SingleScaleAttentionClassifier, 
    SingleScaleMultiHeadClassifier, 
    HierarchicalMultiHeadClassifier
)
from moirai_classification.trainer import get_lsst_dataloaders, get_z_loaders, grid_search_heads, set_seed

# Global Configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_VARS = 6
SIZE = "small"
PATCH_SIZES_TO_TEST = [8, 16, 32, 64]
HEADS_TO_TEST = [4, 8, 16, 32, 64, 128, 384] 

# Hyperparameters
MOIRAI_BATCH_SIZE = 32
HEAD_BATCH_SIZE = 256
LR_GRID = [1e-4]
WD_GRID = [0.05, 0.01]
MODES = ["shared_context", "independent_context"]


df_patch_8_metrics = pd.DataFrame(index=[], columns= ['Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1',
       'Weighted Precision', 'Weighted Recall', 'Weighted F1'])


In [ ]:
set_seed(42)
print("Seed 42 — experiments are reproducible.")

Seed 42 — experiments are reproducible.


In [32]:
raw_tr, raw_va, raw_te, num_classes = get_lsst_dataloaders(batch_size=MOIRAI_BATCH_SIZE, device=DEVICE)

## The Fast Evaluation Loop
Since Moirai is frozen, we loop over the patch sizes, instantiate Moirai, and pre-extract the embeddings ($Z$). We then pass these $Z$ loaders to the different head architectures.

In [33]:
# We define a function to run the evaluation for a specific patch size
def evaluate_heads_for_patch(patch):
    
    # 1. Instantiate the Frozen Encoder
    encoder = MoiraiEncoder(
        module=MoiraiModule.from_pretrained(f"Salesforce/moirai-1.1-R-{SIZE}"),
        prediction_length=patch, context_length=36, patch_size=patch, 
        num_samples=100, target_dim=NUM_VARS, feat_dynamic_real_dim=0, past_feat_dynamic_real_dim=0,
    )
    
    # 2. Pre-Extract Embeddings (Z)
    tr_z, va_z, te_z = get_z_loaders(
        encoder, raw_tr, raw_va, raw_te, 
        head_batch_size=HEAD_BATCH_SIZE, device=DEVICE, remove_last_patch=False, num_vars=NUM_VARS
    )
    
    return tr_z, va_z, te_z

## 1. Mean Pooling Architecture


How it works: The simplest baseline. It takes the contextual patches generated by Moirai and averages them over the temporal dimension. The result is a single flattened vector containing the average representation of each variable, which is then passed to a Linear classifier.

In [34]:
results_mean_pooling = {}

for patch in PATCH_SIZES_TO_TEST:
    tr_z, va_z, te_z = evaluate_heads_for_patch(patch) # Extract once
    
    _, metrics = grid_search_heads(
        MeanPoolingClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes},
        tr_z, va_z, te_z, lr_grid=LR_GRID, wd_grid=WD_GRID, device=DEVICE
    )
    results_mean_pooling[patch] = metrics

In [35]:
df_results_mean_pooling = pd.DataFrame.from_dict(results_mean_pooling, orient='index')
df_results_mean_pooling.index.name = 'Patch Size'
print('results mean pooling')
display(df_results_mean_pooling[['Accuracy']])
df_patch_8_metrics.loc['Mean Pooling'] = df_results_mean_pooling.loc[8]
print('Patch  size = 8')
display(df_patch_8_metrics)

results mean pooling


,Accuracy
Patch Size,
8,0.611517
16,0.618005
32,0.607056
64,0.633414


Patch  size = 8


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Mean Pooling,0.611517,0.454089,0.363178,0.376169,0.563748,0.611517,0.565354


## 2. Single-Scale Attention Architecture


How it works: Instead of a naive average, this head uses a learned Query vector. This Query looks at all the patches (Keys/Values) and assigns an attention weight to each. The network learns which patches are the most important for the classification task.
* shared_context: One single Query acts across all variables simultaneously.
* independent_context: Each variable has its own dedicated Query to find its own important patches.

In [36]:
results_attn = {}

for patch in PATCH_SIZES_TO_TEST:
    tr_z, va_z, te_z = evaluate_heads_for_patch(patch)
    results_attn[patch] = {}

    for mode in MODES:
        _, metrics = grid_search_heads(
            SingleScaleAttentionClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "mode": mode},
            tr_z, va_z, te_z, lr_grid=LR_GRID, wd_grid=WD_GRID, device=DEVICE
        )
        results_attn[patch][mode] = metrics
        if patch == 8:
            df_patch_8_metrics.loc[f"Attention ({mode})"] = metrics

# Accuracy only for all patch sizes
df_attn_acc = pd.DataFrame(
    {patch: {mode: results_attn[patch][mode]['Accuracy'] for mode in MODES} for patch in PATCH_SIZES_TO_TEST}
)
df_attn_acc.index.name = 'Mode'
df_attn_acc.columns.name = 'Patch Size'
print('Results Single-Scale Attention - Accuracy')
display(df_attn_acc.astype(float).round(4))

print('Patch size = 8')
display(df_patch_8_metrics)

Results Single-Scale Attention - Accuracy


Patch Size,8,16,32,64
Mode,,,,
shared_context,0.5969,0.6083,0.5904,0.6298
independent_context,0.6038,0.6095,0.6054,0.6253


Patch size = 8


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Mean Pooling,0.611517,0.454089,0.363178,0.376169,0.563748,0.611517,0.565354
Attention (shared_context),0.596918,0.480749,0.358135,0.372340,0.552259,0.596918,0.548510
Attention (independent_context),0.603812,0.452386,0.355907,0.366543,0.560631,0.603812,0.556651


## 3. Multi-Head Attention (MHA)


How it works: A single attention mechanism might focus entirely on one feature. Multi-Head Attention solves this by projecting the data into multiple sub-spaces.
This allows the model to capture multiple distinct temporal concepts simultaneously.

### 3.1 MHA with 64 heads over all patch sizes

In [37]:
results_mha = {}

for patch in PATCH_SIZES_TO_TEST:
    tr_z, va_z, te_z = evaluate_heads_for_patch(patch)
    results_mha[patch] = {}

    for mode in MODES:
        _, metrics = grid_search_heads(
            SingleScaleMultiHeadClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": 64},
            tr_z, va_z, te_z, lr_grid=LR_GRID, wd_grid=WD_GRID, device=DEVICE
        )
        results_mha[patch][mode] = metrics

# Accuracy only for all patch sizes
df_mha_acc = pd.DataFrame(
    {patch: {mode: results_mha[patch][mode]['Accuracy'] for mode in MODES} for patch in PATCH_SIZES_TO_TEST}
)
df_mha_acc.index.name = 'Mode'
df_mha_acc.columns.name = 'Patch Size'
print('Results MHA (64 heads) - Accuracy')
display(df_mha_acc.astype(float).round(4))

Results MHA (64 heads) - Accuracy


Patch Size,8,16,32,64
Mode,,,,
shared_context,0.6087,0.6277,0.5998,0.633
independent_context,0.6099,0.6217,0.6018,0.629


### 3.2 MHA over patches of sizes 8 but with highter number of heads

In [38]:
results_mha8 = {}

for heads in HEADS_TO_TEST:
    tr_z, va_z, te_z = evaluate_heads_for_patch(8)
    results_mha8[heads] = {}

    for mode in MODES:
        _, metrics = grid_search_heads(
            SingleScaleMultiHeadClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": heads},
            tr_z, va_z, te_z, lr_grid=LR_GRID, wd_grid=WD_GRID, device=DEVICE
        )
        results_mha8[heads][mode] = metrics
        df_patch_8_metrics.loc[f"MHA-{heads} ({mode})"] = metrics

In [39]:
# Accuracy by number of heads (all runs are patch=8)
df_mha8_acc = pd.DataFrame(
    {heads: {mode: results_mha8[heads][mode]['Accuracy'] for mode in MODES} for heads in HEADS_TO_TEST}
)
df_mha8_acc.index.name = 'Mode'
df_mha8_acc.columns.name = 'Num Heads'
print('Results MHA (Patch = 8) - Accuracy by number of heads')
display(df_mha8_acc.astype(float).round(4))

print('Patch size = 8')
display(df_patch_8_metrics)

Results MHA (Patch = 8) - Accuracy by number of heads


Num Heads,4,8,16,32,64,128,384
Mode,,,,,,,
shared_context,0.6139,0.6079,0.6176,0.6115,0.6188,0.6192,0.6184
independent_context,0.6204,0.6135,0.6168,0.6123,0.6119,0.6139,0.6123


Patch size = 8


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Mean Pooling,0.611517,0.454089,0.363178,0.376169,0.563748,0.611517,0.565354
Attention (shared_context),0.596918,0.480749,0.358135,0.372340,0.552259,0.596918,0.548510
Attention (independent_context),0.603812,0.452386,0.355907,0.366543,0.560631,0.603812,0.556651
MHA-4 (shared_context),0.613950,0.415192,0.362670,0.368956,0.554063,0.613950,0.565779
MHA-4 (independent_context),0.620438,0.490441,0.399904,0.420446,0.574098,0.620438,0.583395
MHA-8 (shared_context),0.607867,0.428682,0.376838,0.385446,0.554488,0.607867,0.566619
MHA-8 (independent_context),0.613544,0.437044,0.378955,0.389761,0.565184,0.613544,0.575211
MHA-16 (shared_context),0.617599,0.436923,0.376071,0.387182,0.566021,0.617599,0.574332
MHA-16 (independent_context),0.616788,0.430899,0.375891,0.385644,0.563688,0.616788,0.574367
MHA-32 (shared_context),0.611517,0.432938,0.368723,0.378436,0.559514,0.611517,0.570496


## 4. Hierarchical Multi-Head Attention


How it works: Inspired by Hierarchical Attention Networks (HAN). It performs attention in two steps:
1. Temporal Attention: MHA is applied *within* each variable individually to summarize its temporal patches into a single variable-vector.
2. Variable Attention: A second MHA is applied *across* the summarized variables. This allows the network to learn inter-variable correlations.

In [40]:
results_hier = {}

for patch in PATCH_SIZES_TO_TEST:
    tr_z, va_z, te_z = evaluate_heads_for_patch(patch)
    results_hier[patch] = {}

    for mode in MODES:
        _, metrics = grid_search_heads(
            HierarchicalMultiHeadClassifier, {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": 64},
            tr_z, va_z, te_z, lr_grid=LR_GRID, wd_grid=WD_GRID, device=DEVICE
        )
        results_hier[patch][mode] = metrics
        if patch == 8:
            df_patch_8_metrics.loc[f"Hierarchical MHA ({mode})"] = metrics

df_hier_acc = pd.DataFrame(
    {patch: {mode: results_hier[patch][mode]['Accuracy'] for mode in MODES} for patch in PATCH_SIZES_TO_TEST}
)
df_hier_acc.index.name = 'Mode'
df_hier_acc.columns.name = 'Patch Size'
print('Results Hierarchical MHA (64 heads) - Accuracy')
display(df_hier_acc.astype(float).round(4))

print('Patch size = 8')
display(df_patch_8_metrics)

Results Hierarchical MHA (64 heads) - Accuracy


Patch Size,8,16,32,64
Mode,,,,
shared_context,0.5750,0.5860,0.5819,0.5843
independent_context,0.6115,0.5989,0.5758,0.5852


Patch size = 8


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Mean Pooling,0.611517,0.454089,0.363178,0.376169,0.563748,0.611517,0.565354
Attention (shared_context),0.596918,0.480749,0.358135,0.372340,0.552259,0.596918,0.548510
Attention (independent_context),0.603812,0.452386,0.355907,0.366543,0.560631,0.603812,0.556651
MHA-4 (shared_context),0.613950,0.415192,0.362670,0.368956,0.554063,0.613950,0.565779
MHA-4 (independent_context),0.620438,0.490441,0.399904,0.420446,0.574098,0.620438,0.583395
MHA-8 (shared_context),0.607867,0.428682,0.376838,0.385446,0.554488,0.607867,0.566619
MHA-8 (independent_context),0.613544,0.437044,0.378955,0.389761,0.565184,0.613544,0.575211
MHA-16 (shared_context),0.617599,0.436923,0.376071,0.387182,0.566021,0.617599,0.574332
MHA-16 (independent_context),0.616788,0.430899,0.375891,0.385644,0.563688,0.616788,0.574367
MHA-32 (shared_context),0.611517,0.432938,0.368723,0.378436,0.559514,0.611517,0.570496


## Final Summary
Review of all architectures evaluated on the frozen Moirai encoder.

In [ ]:
print("FINAL SUMMARY — Patch Size = 8 (All Metrics)")
display(df_patch_8_metrics.astype(float).round(4))

print("\nACCURACY ACROSS PATCH SIZES")

print("\nMean Pooling")
display(df_results_mean_pooling[['Accuracy']].astype(float).round(4))

print("\nSingle-Scale Attention")
display(df_attn_acc.astype(float).round(4))

print("\nMHA (64 heads)")
display(df_mha_acc.astype(float).round(4))

print("\nMHA (Patch = 8, varying heads)")
display(df_mha8_acc.astype(float).round(4))

print("\nHierarchical MHA (64 heads)")
display(df_hier_acc.astype(float).round(4))

FINAL SUMMARY — Patch Size = 8 (All Metrics)


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Mean Pooling,0.6115,0.4541,0.3632,0.3762,0.5637,0.6115,0.5654
Attention (shared_context),0.5969,0.4807,0.3581,0.3723,0.5523,0.5969,0.5485
Attention (independent_context),0.6038,0.4524,0.3559,0.3665,0.5606,0.6038,0.5567
MHA-4 (shared_context),0.6139,0.4152,0.3627,0.3690,0.5541,0.6139,0.5658
MHA-4 (independent_context),0.6204,0.4904,0.3999,0.4204,0.5741,0.6204,0.5834
MHA-8 (shared_context),0.6079,0.4287,0.3768,0.3854,0.5545,0.6079,0.5666
MHA-8 (independent_context),0.6135,0.4370,0.3790,0.3898,0.5652,0.6135,0.5752
MHA-16 (shared_context),0.6176,0.4369,0.3761,0.3872,0.5660,0.6176,0.5743
MHA-16 (independent_context),0.6168,0.4309,0.3759,0.3856,0.5637,0.6168,0.5744
MHA-32 (shared_context),0.6115,0.4329,0.3687,0.3784,0.5595,0.6115,0.5705



ACCURACY ACROSS PATCH SIZES

Mean Pooling


,Accuracy
Patch Size,
8,0.6115
16,0.6180
32,0.6071
64,0.6334



Single-Scale Attention


...

Patch Size,8,16,32,64
Mode,,,,
shared_context,0.6087,0.6277,0.5998,0.633
independent_context,0.6099,0.6217,0.6018,0.629



MHA (Patch = 8, varying heads)


Num Heads,4,8,16,32,64,128,384
Mode,,,,,,,
shared_context,0.6139,0.6079,0.6176,0.6115,0.6188,0.6192,0.6184
independent_context,0.6204,0.6135,0.6168,0.6123,0.6119,0.6139,0.6123



Hierarchical MHA (16 heads)


Patch Size,8,16,32,64
Mode,,,,
shared_context,0.5750,0.5860,0.5819,0.5843
independent_context,0.6115,0.5989,0.5758,0.5852


In [43]:
# ── Export Results to CSV ─────────────────────────────────────────────────────
import os
import numpy as np

os.makedirs("results_csv", exist_ok=True)

FINETUNING = "02_Frozen"
MHA_HEADS  = 64
MODE_LABEL = {'shared_context': 'shared', 'independent_context': 'indep'}
rows = []

for patch in PATCH_SIZES_TO_TEST:
    # Mean Pooling (no mode)
    acc = float(df_results_mean_pooling.loc[patch, 'Accuracy'])
    if patch == 8:
        rec = float(df_patch_8_metrics.loc['Mean Pooling', 'Macro Recall'])
        f1  = float(df_patch_8_metrics.loc['Mean Pooling', 'Macro F1'])
    else:
        rec, f1 = float('nan'), float('nan')
    rows.append({'finetuning': FINETUNING, 'pooling': 'Mean', 'patch_size': patch,
                 'accuracy': acc, 'macro_recall': rec, 'macro_f1': f1})

    # Attention — both modes
    for mode, lbl in MODE_LABEL.items():
        acc = float(df_attn_acc.loc[mode, patch])
        if patch == 8:
            rec = float(df_patch_8_metrics.loc[f'Attention ({mode})', 'Macro Recall'])
            f1  = float(df_patch_8_metrics.loc[f'Attention ({mode})', 'Macro F1'])
        else:
            rec, f1 = float('nan'), float('nan')
        rows.append({'finetuning': FINETUNING, 'pooling': f'Attention-{lbl}', 'patch_size': patch,
                     'accuracy': acc, 'macro_recall': rec, 'macro_f1': f1})

    # MHA-64 — both modes (only evaluated at patch=8 in the ablation)
    for mode, lbl in MODE_LABEL.items():
        if patch == 8:
            acc = float(df_mha8_acc.loc[mode, MHA_HEADS])
            rec = float(df_patch_8_metrics.loc[f'MHA-{MHA_HEADS} ({mode})', 'Macro Recall'])
            f1  = float(df_patch_8_metrics.loc[f'MHA-{MHA_HEADS} ({mode})', 'Macro F1'])
        else:
            acc = float(df_mha_acc.loc[mode, patch])
            rec, f1 =  float('nan'), float('nan')
        rows.append({'finetuning': FINETUNING, 'pooling': f'MHA-{lbl}', 'patch_size': patch,
                     'accuracy': acc, 'macro_recall': rec, 'macro_f1': f1})

    # Hierarchical MHA — both modes
    for mode, lbl in MODE_LABEL.items():
        acc = float(df_hier_acc.loc[mode, patch])
        if patch == 8:
            rec = float(df_patch_8_metrics.loc[f'Hierarchical MHA ({mode})', 'Macro Recall'])
            f1  = float(df_patch_8_metrics.loc[f'Hierarchical MHA ({mode})', 'Macro F1'])
        else:
            rec, f1 = float('nan'), float('nan')
        rows.append({'finetuning': FINETUNING, 'pooling': f'Hierarchical-{lbl}', 'patch_size': patch,
                     'accuracy': acc, 'macro_recall': rec, 'macro_f1': f1})

df_nb02 = pd.DataFrame(rows)
df_nb02.to_csv("results_csv/nb02_results.csv", index=False)
print("Saved results_csv/nb02_results.csv")
display(df_nb02)

Saved results_csv/nb02_results.csv


,finetuning,pooling,patch_size,accuracy,macro_recall,macro_f1
0,02_Frozen,Mean,8,0.611517,0.363178,0.376169
1,02_Frozen,Attention-shared,8,0.596918,0.358135,0.372340
2,02_Frozen,Attention-indep,8,0.603812,0.355907,0.366543
3,02_Frozen,MHA-shared,8,0.618816,0.376281,0.388403
4,02_Frozen,MHA-indep,8,0.611922,0.380971,0.396139
5,02_Frozen,Hierarchical-shared,8,0.575020,0.325842,0.315856
6,02_Frozen,Hierarchical-indep,8,0.611517,0.357908,0.358963
7,02_Frozen,Mean,16,0.618005,NaN,NaN
8,02_Frozen,Attention-shared,16,0.608273,NaN,NaN
9,02_Frozen,Attention-indep,16,0.609489,NaN,NaN


In [ ]:
df_patch_8_metrics.to_csv("results_csv/nb02_patch_8_metrics.csv", index=True)